# Session 5: Build your own case

**This part of the workshop needs Navigate installed on your own machine.** If you have not set that up yet, follow the [installation steps in the README](https://github.com/zerocarbonshipping/navigate-zcs/blob/dev-workshop/README.md#installation) first.

Some of the decks in this repository and part of this workshop were written with the help of an AI coding assistant. You can do the same: describe the case you want and let the assistant write a first-shot deck.

This session is about **prompting and running what comes back**. It does not build a
simulation itself (you have to do it).

Put whatever you generate under `simulations/user/`. That folder is gitignored,
so your experiments will not show up as changes to the repository.

> **Read this before you use any of it**
>
> **A prompt gets you a structure, not an answer.** What comes back will run, and
> running is the easy part. Whether it means anything is on you. Each prompt below
> was tested by handing it to an AI assistant with no other instructions and
> running the deck it produced: all three ran without errors on the first attempt.
> But "it runs" is a statement about the syntax, not about the shipping.
>
> Before you interpret a result:
>
> **Know where every input came from.** Open the `.inc` files the assistant wrote.
> A value written as `Forecast("...")` is linked to the assumptions library and
> moves with it. A bare number is not, and needs a comment saying where it came
> from and why the library had none. An assistant will happily produce a deck full
> of plausible numbers, and plausible numbers give plausible, wrong answers.
>
> **Most of the library is a world average, not always your case.** The route is the
> clearest example. All 23 operational profiles in
> `assumptions/defaults/installation/Route/` are defined per *vessel class*, not
> per geography: `operational_profile_tanker_127k_dwt` carries the speeds, time at
> sea and laden/ballast split of the world's Suezmax fleet on average. **None of
> them contains a distance at all**, and the only two ports they know are
> `port_global` and `port_europe` — abstractions for regulatory scope, not places
> on a map. So when your deck says "Ras Tanura to Ningbo", those names and that
> distance are things *you* (or the AI assistant) supplied; the operating pattern underneath is still a
> global average for the class. The same goes for default fuel prices (`fossil_fuel_oil_price_global`), fleet age distributions, orderbooks, and feedstock availability.
>
> **So a case you build here is a structure to think with, not a study of a trade.**

## Three prompts

We propose here three example prompts that lead to a working Navigate deck.
Feel free to take some inspiration from them and create your own!

Each prompt starts with the same short **HOW NAVIGATE WORKS** block. It gives the
assistant what it cannot guess: where the syntax and the data library live, how to
run a deck, and where the results come out as numbers. Keep it when you write your
own prompt. It is what lets any assistant (Claude, Copilot, ChatGPT, ...) work
from this repository rather than from what it thinks a simulation model looks like.

---

### 1. One trade, two ports

Real geography: a named route, a distance, a ship that bunkers at one end or the
other. This one has no regulation at all, so the fuel choice is made on cost
alone.

````text
You are helping me build a simulation deck for Navigate, the shipping
decarbonisation model in this repository. Write the files yourself; do not only
describe them.

HOW NAVIGATE WORKS (read this before writing anything)
- A deck is one .nav file plus the .inc files it includes. The syntax is explained
  in docs/reference_manual/overview.md, and every node type (Fleet, Vessel, Route,
  Port, Fuel, Plant, Producer, Regulation, ...) has its own page in
  docs/reference_manual/.
- assumptions/ is the data library. `Load DefaultVoyage` reads the module file
  assumptions/modules/installation/default_voyage.inc. `Import Fleet "tanker_127k_dwt"`
  reads one node from assumptions/defaults/installation/Fleet/tanker_127k_dwt.inc.
- simulations/examples/example_1 to example_4 are complete decks that run.
- Run a deck from the repository root, in the Python environment where Navigate
  was installed from this repository, with:
      python -m navigate <path to the .nav file> -d ./assumptions
  (`python -m navigate` run from the repository root always uses this
  repository's code; the shorter `navigate` command may point to another copy.)
  Do not install or upgrade any package without asking me first.
- A run writes a .log file next to the .nav. The end of the .log has a "Log
  summary" table with the number of warnings and errors. Some warnings come
  with the library and are expected on any deck built from it (for example "No
  technology cost of capital supplied", "Does not have convex load functions",
  or a feedstock being over-used by a library producer): report them, but do
  not try to fix them. Add `Load DefaultPlot`
  to the .nav DEFINE block to also get the charts in plots/ and plots_analysis/.
  If you cannot run terminal commands, give me the exact command, wait for me to
  paste back the end of the output, and fix whatever it reports.
- To get results as numbers, add `Load DefaultReport` to the .nav DEFINE block.
  The run then also writes plots/<deck name>_report_default.xlsx. Its "Global"
  sheet has one ConsumedEnergy column per fuel (GJ per year; the fuel name is in
  the third header row) and the fleet's emissions (TotalEquivalentWTW).

THE DECK
Build it under simulations/user/gulf_asia_tanker/.

THE CASE
- Fleet: Suezmax-class crude tankers, ~127,000 DWT, on a single trade
- Trade: Ras Tanura (Saudi Arabia) to Ningbo (China), laden out, ballast home
- Fuels: fuel oil and LNG as global commodities, plus e-ammonia and e-methanol
  produced in the Middle East on dedicated renewables
- Regulation: none. Neither port is in the EU. That is the point of the case:
  I want to see what the model does when nothing is pushing it.

- Fleet size: 20 ships to start with (INVENTED; the library's own fleet size for
  this class is the whole world fleet)

The fleet starts on fuel oil. The model decides whether to retrofit to dual-fuel,
replace at end of life, or keep burning oil.

HOW TO BUILD IT
- Build your own Fleet node from the library's Vessel files with Copy Vessel, as
  simulations/examples/example_4/includes/fleet.inc does. Import Fleet would bring
  the world fleet, with a world-sized fleet and orderbook.
- The library's operational profile is a world-average pattern between two
  abstract ports (port_global, port_europe). Do not use those ports: build a
  ROUND_TRIP route between the two named ports, taking speeds and time in port
  from the profile's own values, as example_4's route.inc does (its comments show
  the arithmetic). Use a realistic sea distance and mark it INVENTED.
- Rebuild the other parts example_4 has, with this case's own values: the ports,
  the bunker logistics (how fuel travels from the plants to each port) and the
  regions (where the fuels are made).
- Give the fuel producers a share of the library's global feedstock ceilings,
  marked INVENTED, as example_4's producer.inc does.
- Energy-efficiency technologies are optional here: leave them out.

NUMBERS
Every number must come from assumptions/ and be referenced by name
(Forecast("..."), Curve("..."), Variable("...")), never copied in as a literal.
Where the library itself stores a plain number (the fuel-conversion costs in a
Fleet file, the time at sea in a Route file), copy it with a comment naming the
file it came from.
The library already covers this vessel class - use its own values:
    assumptions/defaults/installation/Fleet/tanker_127k_dwt.inc
    assumptions/defaults/installation/Route/operational_profile_tanker_127k_dwt.inc
Take structure from simulations/examples/example_4/ and none of its numbers.
Anything the library genuinely lacks - port-to-port distance or the bunker
transport cost, for instance - mark INVENTED in a comment where it appears and
list them in the .nav header. BunkerLogistics takes its distances and transport
cost as plain numbers, not as Variables.

DECK RULES
The .nav DEFINE and EVENTS blocks accept only Include and Load. Every node
declaration, Import and Copy goes in an .inc file that they include.

BEFORE YOU TELL ME IT WORKS
Run it (or have me run it) as above. Report the exit code and the warning count
from the Log summary. Tell me the fuel mix in the first year and the last, from
the report, and say plainly whether anything actually switched.
````

---

### 2. The world fleet, simplified

No geography here: the Center's reference scenarios work this way, with twenty-three
vessel classes on the library's own global operating profiles, and the only
"places" are two abstract ports that exist so the EU measures know what they
cover. This cuts that down to a size you can iterate on.

````text
You are helping me build a simulation deck for Navigate, the shipping
decarbonisation model in this repository. Write the files yourself; do not only
describe them.

HOW NAVIGATE WORKS (read this before writing anything)
- A deck is one .nav file plus the .inc files it includes. The syntax is explained
  in docs/reference_manual/overview.md, and every node type (Fleet, Vessel, Route,
  Port, Fuel, Plant, Producer, Regulation, ...) has its own page in
  docs/reference_manual/.
- assumptions/ is the data library. `Load DefaultVoyage` reads the module file
  assumptions/modules/installation/default_voyage.inc. `Import Fleet "tanker_127k_dwt"`
  reads one node from assumptions/defaults/installation/Fleet/tanker_127k_dwt.inc.
- simulations/scenarios/basecase_mid_regulation/ is the full reference scenario
  this case cuts down; its .nav is almost entirely Load statements.
- Run a deck from the repository root, in the Python environment where Navigate
  was installed from this repository, with:
      python -m navigate <path to the .nav file> -d ./assumptions
  (`python -m navigate` run from the repository root always uses this
  repository's code; the shorter `navigate` command may point to another copy.)
  Do not install or upgrade any package without asking me first.
- A run writes a .log file next to the .nav. The end of the .log has a "Log
  summary" table with the number of warnings and errors. Some warnings come
  with the library and are expected on any deck built from it (for example "No
  technology cost of capital supplied", "Does not have convex load functions",
  or a feedstock being over-used by a library producer): report them, but do
  not try to fix them. Add `Load DefaultPlot`
  to the .nav DEFINE block to also get the charts in plots/ and plots_analysis/.
  If you cannot run terminal commands, give me the exact command, wait for me to
  paste back the end of the output, and fix whatever it reports.
- To get results as numbers, add `Load DefaultReport` to the .nav DEFINE block.
  The run then also writes plots/<deck name>_report_default.xlsx. Its "Global"
  sheet has one ConsumedEnergy column per fuel (GJ per year; the fuel name is in
  the third header row) and the fleet's emissions (TotalEquivalentWTW).

THE DECK
Build it under simulations/user/[YOUR FOLDER NAME]/.

THE CASE
A cut-down version of the Center's reference scenarios. No route, no map, no real
ports - I want the global picture, not a trade.

- Fleets: [TWO OR THREE CLASSES, e.g. tanker_127k_dwt and bulk_carrier_80k_dwt],
  imported whole from the library (Import Fleet "..."), so each brings its own
  vessels, age distribution, orderbook, technologies and retrofit costs
- Geography: none of your own. Load DefaultVoyage for the per-class operating
  profiles and DefaultPort for the two abstract ports the library uses.
- Fuels and production: load the library's whole chain - DefaultEmission,
  DefaultFeedstock, DefaultFuel, DefaultProducer, DefaultPlant, DefaultRegion,
  DefaultSource, DefaultTransport
- Regulation: EU ETS and FuelEU at their library settings (DefaultRegulation)
- Also load what every library deck needs, as the reference scenario's .nav does:
  DefaultModelDefinition, DefaultBunkerLogistics (without it the first year's
  bunkering has no feasible solution) and DefaultCalibration in DEFINE, and
  DefaultTimeStepYearly in EVENTS

NUMBERS
Everything comes from the library by construction here, because the deck is
almost entirely Load statements. If you find yourself writing a number, stop and
tell me why the library has none.

DECK RULES
- The .nav DEFINE and EVENTS blocks accept only Include and Load. Import Fleet
  statements go in an .inc file that DEFINE includes.
- Do NOT load DefaultConverterAvailable. It phases alternative vessel types in
  over time using wildcards like Fleet "tug*" that only resolve against the full
  twenty-three-fleet set, so it fails to parse on a cut-down deck. Say in the
  .nav header that every vessel type is therefore orderable from year one rather
  than phased in.

BEFORE YOU TELL ME IT WORKS
Run it (or have me run it) as above - expect three to four minutes, not seconds. Report
the exit code and the warning count from the Log summary, and tell me the fuel
mix at both ends of the horizon, from the report.
````

---

### 3. Two fleets, one market

No new geography either: the same trade twice, run by two fleets whose owners
make different choices.

````text
You are helping me build a simulation deck for Navigate, the shipping
decarbonisation model in this repository. Write the files yourself; do not only
describe them.

HOW NAVIGATE WORKS (read this before writing anything)
- A deck is one .nav file plus the .inc files it includes. The syntax is explained
  in docs/reference_manual/overview.md, and every node type (Fleet, Vessel, Route,
  Port, Fuel, Plant, Producer, Regulation, ...) has its own page in
  docs/reference_manual/.
- assumptions/ is the data library. `Load DefaultVoyage` reads the module file
  assumptions/modules/installation/default_voyage.inc. `Import Fleet "tanker_127k_dwt"`
  reads one node from assumptions/defaults/installation/Fleet/tanker_127k_dwt.inc.
- simulations/examples/example_1 to example_4 are complete decks that run.
- Run a deck from the repository root, in the Python environment where Navigate
  was installed from this repository, with:
      python -m navigate <path to the .nav file> -d ./assumptions
  (`python -m navigate` run from the repository root always uses this
  repository's code; the shorter `navigate` command may point to another copy.)
  Do not install or upgrade any package without asking me first.
- A run writes a .log file next to the .nav. The end of the .log has a "Log
  summary" table with the number of warnings and errors. Some warnings come
  with the library and are expected on any deck built from it (for example "No
  technology cost of capital supplied", "Does not have convex load functions",
  or a feedstock being over-used by a library producer): report them, but do
  not try to fix them. Add `Load DefaultPlot`
  to the .nav DEFINE block to also get the charts in plots/ and plots_analysis/.
  If you cannot run terminal commands, give me the exact command, wait for me to
  paste back the end of the output, and fix whatever it reports.
- To get results as numbers, add `Load DefaultReport` to the .nav DEFINE block.
  The run then also writes plots/<deck name>_report_default.xlsx, with totals for
  both fleets together. For results per fleet, add your own Report node as well,
  with the add_fleet_property / add_vessel_property syntax of
  simulations/examples/example_4/includes/report.inc (copy the syntax, not its
  list): per fleet FuelTypeEnergy, TotalEquivalentWTW, FuelExpenses,
  RegulationExpenses, and per vessel InvestmentFreightRate.

THE DECK
Build it under simulations/user/[YOUR FOLDER NAME]/.

THE CASE
Two fleets in the same market, so I can see how much the owners' choices matter
rather than the trade.

- Trade: [ORIGIN] to [DESTINATION], one route, both fleets sailing it
- Fleet A: [VESSEL CLASS], no energy-efficiency measures fitted
- Fleet B: the same class and the same trade, but fitting the efficiency
  technologies the library offers for it
- Fuels: [WHICH ONES SHOULD COMPETE]
- Regulation: [EU ETS AND FUELEU / NONE]
- Fleet size: [N] ships each to start with (INVENTED)

Model them as two Fleet nodes sharing one Route. simulations/examples/example_3/
does this for two vessel classes (so four fleets): follow its pattern of one
"regular" and one "efficient" fleet, for this one class. Same costs, same route,
same regulation - the only difference is what they fit. Fleet A fits nothing at
all: leave out the Fleet file's set_initial_technology_share lines for it and set
AllowTechnologyApproximation = FALSE, as example_3's regular fleet does.

As in example_4, build the Fleet nodes from the library's Vessel files with Copy
Vessel, and build a ROUND_TRIP route between the two named ports from the class's
operational profile (not its abstract port_global / port_europe ports; example_4's
route.inc comments show the arithmetic), with a realistic sea distance marked
INVENTED. Rebuild the ports, bunker logistics, regions and producers the same way,
and use `Load DefaultCalibration` for the decision settings. If you rename the
copied vessels, check that the Fleet file's set_fuel_conversion_cost wildcards
still match the new names, and restate those lines if they do not.

NUMBERS
Every number must come from assumptions/ and be referenced by name
(Forecast("..."), Curve("..."), Variable("...")), never copied in as a literal.
Where the library itself stores a plain number (the fuel-conversion costs in a
Fleet file, the time at sea in a Route file), copy it with a comment naming the
file it came from.
Use the library's own Fleet file for the class, including its technology list and
retrofit costs. Take structure from example_3 and example_4 and none of their
numbers.
Mark anything INVENTED in a comment and list it in the .nav header.

DECK RULES
The .nav DEFINE and EVENTS blocks accept only Include and Load. Every node
declaration, Import and Copy goes in an .inc file that they include.

BEFORE YOU TELL ME IT WORKS
Run it (or have me run it) as above. Report the exit code and the warning count
from the Log summary. Then tell me what separates the two fleets by 2050 - fuel
mix, emissions and the freight rate - and how much fuel and compliance cost the
efficient fleet saved. This version of Navigate does not report what the
measures themselves cost, so say that rather than concluding they paid for
themselves.
````

---

### Bonus: a prompt for writing prompts

If you are not sure what to ask for, hand this over first and let it interview
you.

````text
I want to build a simulation with Navigate, the shipping decarbonisation model in
this repository. Do not write any deck yet. Interview me first.

Read these so your questions are informed rather than generic (if you cannot open
files in this repository, tell me and I will paste them):
  docs/reference_manual/overview.md          how a deck is put together
  simulations/examples/                      four worked cases of different shapes
  assumptions/defaults/installation/Fleet/   which vessel classes exist already

Then ask me, a few questions at a time: what question I actually want answered;
which fleet, and whether the library already covers that class; whether geography
matters for my question or whether a global run would do; which fuels should
compete; and which regulation applies, if any.

When you have enough, write out the prompt you would want to be given, starting
with a short "how Navigate works" section like the one in the prompts above, show
it to me, and wait for me to approve it before building anything.
````

## Running what you get

In a terminal, in the Python environment where you installed Navigate, run from
the repository root:

```bash
python -m navigate simulations/user/<your_case>/<your_case>.nav -d ./assumptions
```

Three things worth knowing:

- **`python -m navigate` from the repository root always runs this repository's
  code.** The shorter `navigate simulations/...` works too, but if another copy of
  Navigate is installed on your machine it may run that one instead, and fail on
  decks that are fine.

- **`-d ./assumptions` is needed** whenever the deck uses `Load`, `Import` or
  `Copy` — which any deck built on the library does. A fully self-contained deck
  like the minimal single-vessel example of [Tutorial 1](https://zerocarbonshipping.github.io/navigate-zcs/tutorials/tutorial_1.html) needs no `-d` at all.
- **Output lands next to the deck**, not in your working directory. Navigate
  resolves the deck path first and writes everything relative to that.

### Where the plots are

`Load DefaultPlot` declares two `Plot` nodes, so a run fills two folders inside
the deck's own directory:

| where | what is in it |
|---|---|
| `<deck folder>/plots/` | the headline set, nine charts — among them fuel consumed, absolute emissions, installed power, producer development |
| `<deck folder>/plots_analysis/` | the fuller set, thirty-two charts — among them fleet evolution and conversions, speeds, emission intensity, expenses, bunker price and supply per port, plant production costs |

How many you get depends on what the deck contains. Two more files land beside
them: `<deck>.log`, and `plot_data.pkl`, which holds the solved model.

Open the PNGs from the file browser, or in a notebook with
`from IPython.display import Image; display(Image(filename=...))`. Which charts
appear is set by the `add_plot("...")` calls in the deck; the full list of labels
is in the appendix of the reference manual's
[Plot page](https://zerocarbonshipping.github.io/navigate-zcs/reference_manual/plot.html).

> **Old plots are not cleared**
>
> Navigate overwrites a chart it regenerates but never deletes one it no longer
> produces, so a PNG from an earlier configuration can sit in the folder looking
> exactly as current as the rest. Delete the folder before a re-run if you have
> changed which plots the deck asks for.

If the run finishes without errors, check the `Log summary` table at the foot of
`<deck>.log`: it counts warnings and errors, and is the quickest way to tell a
clean run from one that merely completed.

## Workshop sessions

This is the last session of the workshop.

Each link opens the notebook in Google Colab, in a fresh session, so run it
from the top.

1. [Run a simple case](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/01-run-a-simple-case.ipynb)
2. [Understand Navigate](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/02-understand-navigate.ipynb)
3. [Run a global case](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/03-run-a-global-case.ipynb)
4. [Run our reference scenarios](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/04-run-our-reference-scenarios.ipynb)
5. **Build your own case** (this session)